In [1]:
import logging
import json
import numpy as np
import inspect
import numpy as np
import pandas as pd

import random
from typing import Dict, List, Optional, Tuple, Any

from pathlib import Path
from typing import Optional, Dict, Any

from utils.paths import get_paths
from utils.file_io import save_data
from utils.logging_setup import configure_logging, log_layer_paths
from utils.pipeline_config_loader import (
    load_pipeline_config,
    set_wandb_dir_from_config,
    export_config_snapshot,
)

from utils.truths import (
    make_process_run_id,
    initialize_layer_truth,
    update_truth_section,
    build_truth_record,
    save_truth_record,
    append_truth_index,
    stamp_truth_columns,
    load_truth_record_by_hash,
    get_truth_hash,
    get_parent_truth_hash,
    get_pipeline_mode_from_truth,
    get_artifact_path_from_truth,
)

from utils.synthetic.generator.synthetic_profiles import (
    load_rich_profile_csv,
    load_and_merge_rich_profiles,
    load_correlation_pairs_csv,
    load_group_map_csv,
    load_fault_pairings_csv,
)


from utils.synthetic.generator.synthetic_missingness import build_missingness_spec_from_truth_payload


from utils.synthetic.generator.synthetic_generator import SyntheticGenerator, EpisodeSpec

from utils.synthetic.generator.synthetic_postgres_writer import (
    ensure_sequence,
    reserve_next_batch_id,
    reserve_cycle_range,
    reset_sequence,
    reset_synthetic_sequences,
    write_stream_batch,
)

from utils.synthetic.generator.synthetic_export import export_synthetic_batch_to_parquet
from utils.pipeline_config_loader import build_truth_config_block

from utils.postgres_util import get_engine_from_env, build_postgres_url, execute_sql, read_sql_dataframe
from utils.layer_postgres_writer import write_layer_dataframe, prepare_layer_dataframe


In [2]:
# --- Notebook params ---
STAGE = "synthetic"
DATASET = "pump"
MODE = "train"
PROFILE = "default"

In [3]:
paths = get_paths()

config_obj = load_pipeline_config(
    config_root=paths.configs,
    stage=STAGE,
    dataset=DATASET,
    mode=MODE,
    profile=PROFILE,
    project_root=paths.root,
)
CONFIG = config_obj.data

SYN_CFG = CONFIG["synthetic"]
PATHS = CONFIG["resolved_paths"]
PIPELINE = CONFIG.get("pipeline", {"execution_mode": "batch", "orchestration_mode": "notebook"})

PIPELINE_MODE = PIPELINE["execution_mode"]
DATASET_NAME = str(CONFIG["dataset"]["name"]).strip().lower()
TRUTH_VERSION = CONFIG["versions"]["truth"]

TRUTHS_PATH = Path(PATHS["truths_dir"])
TRUTH_INDEX_PATH = Path(PATHS["truth_index_path"])
LOGS_PATH = Path(PATHS["logs_root"])
ARTIFACTS_ROOT = Path(PATHS["artifacts_root"])


TRUTHS_PATH.mkdir(parents=True, exist_ok=True)
LOGS_PATH.mkdir(parents=True, exist_ok=True)
ARTIFACTS_ROOT.mkdir(parents=True, exist_ok=True)



set_wandb_dir_from_config(CONFIG)

print("DATASET_NAME:", DATASET_NAME)
print("TRUTHS_PATH:", TRUTHS_PATH)
print("ARTIFACTS_ROOT:", ARTIFACTS_ROOT)

DATASET_NAME: pump
TRUTHS_PATH: /workspace/artifacts/truths
ARTIFACTS_ROOT: /workspace/artifacts


In [4]:
# Logging Setup

# Create gold log path 
synthetic_log_path = paths.logs / "synthetic_data_generator.log"

# Initial Logger
configure_logging(
    "capstone",
    synthetic_log_path,
    level=logging.DEBUG,
    overwrite_handlers=True,
)

# Initiate Logger and log file
logger = logging.getLogger("capstone.synthetic")

# Log load and initiation
logger.info("Synethetic Data Generation starting")

# Log paths loads
log_layer_paths(paths, current_layer="synthetic", logger=logger)


2026-04-05 07:04:32,265 | INFO | capstone.synthetic | Synethetic Data Generation starting
2026-04-05 07:04:32,267 | INFO | capstone.synthetic | Project Root Path Loaded: /workspace
2026-04-05 07:04:32,269 | INFO | capstone.synthetic | Project Logging Path Loaded: /workspace/logs
2026-04-05 07:04:32,270 | INFO | capstone.synthetic | Project Artifacts Path Loaded: /workspace/artifacts
2026-04-05 07:04:32,272 | INFO | capstone.synthetic | Project Notebooks Path Loaded: /workspace/notebooks
2026-04-05 07:04:32,273 | INFO | capstone.synthetic | Project Truths Path Loaded: /workspace/artifacts/truths
2026-04-05 07:04:32,275 | INFO | capstone.synthetic | Project Data Path Loaded: /workspace/data
2026-04-05 07:04:32,276 | INFO | capstone.synthetic | Previous Layer (Gold) Path Loaded: /workspace/data/gold


In [5]:
def sample_int_range(rng: np.random.Generator, value_or_range, *, low_inclusive: bool = True) -> int:
    """
    Accepts either:
      - int
      - [low, high]
    Returns an int.
    """
    if isinstance(value_or_range, (int, np.integer)):
        return int(value_or_range)

    if isinstance(value_or_range, (list, tuple)) and len(value_or_range) == 2:
        low = int(value_or_range[0])
        high = int(value_or_range[1])
        if low_inclusive:
            # numpy integers are low inclusive, high exclusive
            return int(rng.integers(low, high + 1))
        return int(rng.integers(low, high))

    raise TypeError(f"Expected int or [low, high] range, got: {type(value_or_range)} -> {value_or_range}")


def sample_float_range(rng: np.random.Generator, value_or_range) -> float:
    """
    Accepts either:
      - float/int
      - [low, high]
    Returns a float.
    """
    if isinstance(value_or_range, (float, int, np.floating, np.integer)):
        return float(value_or_range)

    if isinstance(value_or_range, (list, tuple)) and len(value_or_range) == 2:
        low = float(value_or_range[0])
        high = float(value_or_range[1])
        return float(rng.uniform(low, high))

    raise TypeError(f"Expected float or [low, high] range, got: {type(value_or_range)} -> {value_or_range}")

In [6]:
# Get Latest Truth Hash

def get_latest_truth_hash(
    *,
    truth_index_path: Path,
    layer_name: str,
    dataset_name: str,
) -> str:
    """
    Return the most recent truth_hash for a given layer + dataset from truth_index.jsonl.

    Assumes truth_index.jsonl is append-only and newer entries are later in the file.
    """
    if not truth_index_path.exists():
        raise FileNotFoundError(f"truth_index.jsonl not found: {truth_index_path}")

    dataset_name_norm = str(dataset_name).strip().lower()
    layer_name_norm = str(layer_name).strip().lower()

    latest_record: Optional[Dict[str, Any]] = None

    with truth_index_path.open("r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if not line:
                continue

            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue

            rec_layer = str(rec.get("layer_name", "")).strip().lower()
            rec_dataset = str(rec.get("dataset_name", "")).strip().lower()

            if rec_layer == layer_name_norm and rec_dataset == dataset_name_norm:
                latest_record = rec

    if latest_record is None:
        raise ValueError(
            f"No truth records found for layer='{layer_name}' dataset='{dataset_name}' in {truth_index_path}"
        )

    truth_hash = latest_record.get("truth_hash")
    if truth_hash is None or str(truth_hash).strip() == "":
        raise ValueError("Latest record is missing a usable truth_hash.")

    return str(truth_hash).strip()

#### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### #### 


def get_latest_silver_eda_truth_hash(
    *,
    truth_index_path: Path,
    dataset_name: str,
) -> str:
    """
    Convenience wrapper: latest Silver EDA truth hash for a dataset.
    """
    return get_latest_truth_hash(
        truth_index_path=truth_index_path,
        layer_name="silver_eda",
        dataset_name=dataset_name,
    )

In [7]:
# Updated
# --- Notebook params ---
STAGE = SYN_CFG["layer_name"]
DATASET = "pump"
MODE = "train"
PROFILE = "default"

# Parent truth hash from your latest Silver EDA run
SILVER_EDA_TRUTH_HASH = get_latest_silver_eda_truth_hash(truth_index_path=TRUTH_INDEX_PATH, dataset_name=DATASET_NAME,)
print("Latest SILVER_EDA_TRUTH_HASH:", SILVER_EDA_TRUTH_HASH)

# Faults
# Episode overrides (easy test knobs)
PRIMARY_SENSOR = None          # None => first sensor
PRIMARY_FAULT_TYPE = list(SYN_CFG["faults"]["allowed"])



# Episode Settings
NORMAL_BEFORE = list(SYN_CFG["episode"]["normal_before_range"])
BUILDUP = list(SYN_CFG["episode"]["buildup_range"])
FAILURE = list(SYN_CFG["episode"]["failure_range"])
RECOVERY = list(SYN_CFG["episode"]["recovery_range"])
NORMAL_AFTER = list(SYN_CFG["episode"]["normal_after_range"])
MAGNITUDE = list(SYN_CFG["episode"]["magnitude_range"])

SYNTH_PROCESS_RUN_ID = make_process_run_id(SYN_CFG["process_run_id_prefix"])

# Outputs
OUTPUT_MODE = SYN_CFG["output_mode"]

# Postgres settings
PG_SCHEMA = str(SYN_CFG["postgres"]["schema"])
TABLE_ARTIFACT_NAME = str(SYN_CFG["postgres"]["table_artifact_name"])

# medallion naming: synthetic_<dataset>_<artifact_name>
ARTIFACT_NAME = "stream"       

# Export
EXPORT_ENABLED = bool(SYN_CFG["export"]["enabled"])
EXPORT_DIRECTORY = str(SYN_CFG["export"]["export_dir_key"])

Latest SILVER_EDA_TRUTH_HASH: a338fe683b00924e9cab15aeb4bdeb911c2c12b6c65e053c91befd06dcee8090


----

In [8]:
# ---- Mode switch ----
MODE = "batch"         # "single" | "batch"
TARGET_ROWS = 72_000
MAX_EPISODES = 1_000_000  # safety cap

# ---- policy knobs ----
EPISODE_MAX_ROWS = 3_000   # prevents monster episodes; forces multiple episodes in a 10k batch
ALLOW_OVERSHOOT = False    # if True, can overshoot when remaining can't fit minimum core

# ---- failure rarity control (match real dataset frequency) ----
# Real dataset: ~7 failures per 250,000 rows  => ~1 failure per 35,714 rows
ROWS_PER_FAILURE = 250_000 / 7  # ~35714.2857



PG_SCHEMA = "capstone"
ARTIFACT_NAME = "stream"
TABLE_NAME = f"synthetic_{DATASET_NAME.lower()}_{ARTIFACT_NAME}"

# ---- write mode flags
WRITE_MODE = "reset"       # "reset" | "append"
APPEND_MODE = "renumber"    # "continue" | "renumber"   (only matters if WRITE_MODE="append")


---

In [9]:
if SILVER_EDA_TRUTH_HASH is None or str(SILVER_EDA_TRUTH_HASH).strip() == "":
    raise ValueError("Set SILVER_EDA_TRUTH_HASH in the parameter cell.")

silver_eda_truth = load_truth_record_by_hash(
    truth_dir=TRUTHS_PATH,
    layer_name="silver_eda",
    dataset_name=DATASET_NAME,
    truth_hash=str(SILVER_EDA_TRUTH_HASH).strip(),
)


PARENT_TRUTH_HASH = get_truth_hash(silver_eda_truth)

SILVER_PREEDA_TRUTH_HASH = get_parent_truth_hash(silver_eda_truth)

silver_preeda_truth = load_truth_record_by_hash(
    truth_dir=TRUTHS_PATH,
    layer_name="silver",
    dataset_name=DATASET_NAME,
    truth_hash=str(SILVER_PREEDA_TRUTH_HASH).strip(),
)

missingness_payload = (silver_preeda_truth.get("runtime_facts", {}) or {}).get("missingness_quarantine", {}) or {}
missingness_spec = build_missingness_spec_from_truth_payload(missingness_payload)


parent_mode = get_pipeline_mode_from_truth(silver_eda_truth)
if parent_mode:
    PIPELINE_MODE = parent_mode

print("PARENT_TRUTH_HASH:", PARENT_TRUTH_HASH)
print("PIPELINE_MODE:", PIPELINE_MODE)

logger.info("W&B PARENT_TRUTH_HASH: %s", PARENT_TRUTH_HASH)
logger.info("W&B PIPELINE_MODE: %s", PIPELINE_MODE)

2026-04-05 07:04:32,437 | INFO | capstone.synthetic | W&B PARENT_TRUTH_HASH: a338fe683b00924e9cab15aeb4bdeb911c2c12b6c65e053c91befd06dcee8090
2026-04-05 07:04:32,439 | INFO | capstone.synthetic | W&B PIPELINE_MODE: batch


PARENT_TRUTH_HASH: a338fe683b00924e9cab15aeb4bdeb911c2c12b6c65e053c91befd06dcee8090
PIPELINE_MODE: batch


In [10]:
keys = SYN_CFG["silver_eda_artifact_keys"]

profile_normal_path = get_artifact_path_from_truth(silver_eda_truth, keys["profile_normal"])
profile_abnormal_path = get_artifact_path_from_truth(silver_eda_truth, keys["profile_abnormal"])
profile_recovery_path = get_artifact_path_from_truth(silver_eda_truth, keys["profile_recovery"])

corr_pairs_normal_path = get_artifact_path_from_truth(silver_eda_truth, keys["corr_pairs_normal"])
group_map_normal_path = get_artifact_path_from_truth(silver_eda_truth, keys["group_map_normal"])
fault_pairings_normal_path = get_artifact_path_from_truth(silver_eda_truth, keys["fault_pairings_normal"])

print(profile_normal_path)
print(profile_abnormal_path)
print(profile_recovery_path)
print(corr_pairs_normal_path)
print(group_map_normal_path)
print(fault_pairings_normal_path)

logger.info("silver_eda_artifact_keys: %s", keys)


2026-04-05 07:04:32,455 | INFO | capstone.synthetic | silver_eda_artifact_keys: {'profile_normal': 'feature_profile_normal_path', 'profile_abnormal': 'feature_profile_abnormal_path', 'profile_recovery': 'feature_profile_recovery_path', 'corr_pairs_normal': 'sensor_correlation_pairs_normal_path', 'group_map_normal': 'sensor_group_map_normal_path', 'fault_pairings_normal': 'sensor_fault_pairings_normal_path'}


/workspace/artifacts/silver_eda/pump/pump__silver_eda__feature_profile__normal.csv
/workspace/artifacts/silver_eda/pump/pump__silver_eda__feature_profile__abnormal.csv
/workspace/artifacts/silver_eda/pump/pump__silver_eda__feature_profile__recovery.csv
/workspace/artifacts/silver_eda/pump/sensor_correlation_pairs_normal.csv
/workspace/artifacts/silver_eda/pump/sensor_group_map_normal.csv
/workspace/artifacts/silver_eda/pump/sensor_fault_pairings_normal.csv


In [11]:
def load_episode_status_counts_json(path: str | Path) -> list[dict]:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"episode_status_counts.json not found: {p}")
    data = json.loads(p.read_text())
    if not isinstance(data, list):
        raise ValueError("episode_status_counts.json must be a JSON list of objects")
    return data


EPISODE_STATUS_JSON_PATH = str(ARTIFACTS_ROOT / f"silver_eda/{DATASET_NAME}/{DATASET_NAME}__silver_eda__episode_status_counts.json")

episode_status_rows = load_episode_status_counts_json(EPISODE_STATUS_JSON_PATH)
print("Loaded episodes:", len(episode_status_rows))
print("First row:", episode_status_rows[0] if episode_status_rows else None)

Loaded episodes: 8
First row: {'meta__episode_id': 0, 'normal': 17155, 'normal_percent': 0.9477900552486188, 'abnormal': 1, 'abnormal_percent': 5.524861878453039e-05, 'recovery': 944, 'recovery_percent': 0.052154696132596684, 'episode_total_rows': 18100}


In [12]:
def summarize_episode_status_rows(rows: list[dict]) -> dict:
    if not rows:
        raise ValueError("No episode status rows provided")

    normals   = np.array([float(r.get("normal", 0)) for r in rows], dtype=float)
    failures  = np.array([float(r.get("failure", 0)) for r in rows], dtype=float)
    recoveries= np.array([float(r.get("recovery", 0)) for r in rows], dtype=float)
    totals    = np.array([float(r.get("episode_total_rows", 0)) for r in rows], dtype=float)

    # percents (if present, prefer those; else compute)
    rec_p = []
    for r in rows:
        if "recovery_percent" in r:
            rec_p.append(float(r["recovery_percent"]))
        else:
            t = float(r.get("episode_total_rows", 0)) or 1.0
            rec_p.append(float(r.get("recovery", 0)) / t)
    rec_p = np.array(rec_p, dtype=float)

    fail_p = []
    for r in rows:
        if "failure_percent" in r:
            fail_p.append(float(r["failure_percent"]))
        else:
            t = float(r.get("episode_total_rows", 0)) or 1.0
            fail_p.append(float(r.get("failure", 0)) / t)
    fail_p = np.array(fail_p, dtype=float)

    # dataset-faithful defaults (robust)
    summary = {
        "episodes_n": int(len(rows)),

        # Episode sizing
        "episode_total_median": int(np.median(totals)),
        "episode_total_p10": int(np.quantile(totals, 0.10)),
        "episode_total_p90": int(np.quantile(totals, 0.90)),

        # Failure is a marker in your dataset
        "failure_rows_default": int(round(np.median(failures))),  # should be 1
        "failure_rows_max": int(np.max(failures)),

        # Recovery variability is big in your sample — keep it learnable
        "recovery_percent_median": float(np.median(rec_p)),
        "recovery_percent_mean": float(np.mean(rec_p)),
        "recovery_percent_p10": float(np.quantile(rec_p, 0.10)),
        "recovery_percent_p90": float(np.quantile(rec_p, 0.90)),

        # Normal is whatever remains
        "normal_percent_median": float(np.median(normals / np.where(totals == 0, 1, totals))),
    }

    return summary

episode_shape_summary = summarize_episode_status_rows(episode_status_rows)
print("Episode shape summary:")
for k, v in episode_shape_summary.items():
    print(f"  {k}: {v}")

Episode shape summary:
  episodes_n: 8
  episode_total_median: 21721
  episode_total_p10: 6858
  episode_total_p90: 55073
  failure_rows_default: 0
  failure_rows_max: 0
  recovery_percent_median: 0.04132961498096935
  recovery_percent_mean: 0.08042409713908762
  recovery_percent_p10: 0.0020715779505188813
  recovery_percent_p90: 0.1991914614035746
  normal_percent_median: 0.9586311355061974


In [13]:
def choose_episode_phase_lengths(
    rng: np.random.Generator,
    *,
    episode_status_rows: list[dict],
    # knobs
    failure_rows_default: int = 1,
    # how much of "normal" becomes buildup
    buildup_fraction_of_normal: float = 0.10,   # 10% of normal reserved for buildup window
    min_buildup_rows: int = 10,
    min_normal_before_rows: int = 50,
    min_normal_after_rows: int = 50,
    # choose episode total size from the real distribution
    use_real_episode_totals: bool = True,
) -> dict:
    """
    Returns dict with:
      normal_before, buildup, failure, recovery, normal_after, episode_total
    """

    if not episode_status_rows:
        raise ValueError("episode_status_rows is empty")

    # choose a "template" episode from real distribution
    template = episode_status_rows[int(rng.integers(0, len(episode_status_rows)))]
    real_total = int(template.get("episode_total_rows", 0))

    if use_real_episode_totals and real_total > 0:
        episode_total = real_total
    else:
        # fallback: derive from the JSON distribution
        totals = [int(r.get("episode_total_rows", 0)) for r in episode_status_rows if int(r.get("episode_total_rows", 0)) > 0]
        episode_total = int(rng.choice(totals)) if totals else 5000

    # failure (dataset-faithful)
    failure = int(failure_rows_default)

    # recovery: use template recovery_percent if present
    rec_pct = template.get("recovery_percent", None)
    if rec_pct is None:
        rec = int(template.get("recovery", 0))
    else:
        rec = int(round(float(rec_pct) * episode_total))

    # clamp recovery to something valid
    rec = max(0, min(episode_total - failure, rec))

    remaining = episode_total - failure - rec
    if remaining < 0:
        # fallback if episode_total too small
        remaining = 0
        rec = max(0, episode_total - failure)

    # split remaining into normal + buildup
    # buildup is a fraction of remaining normal-ish region
    buildup = int(round(remaining * float(buildup_fraction_of_normal)))
    buildup = max(min_buildup_rows, buildup) if remaining >= min_buildup_rows else min(remaining, min_buildup_rows)

    normal_total = max(0, remaining - buildup)

    # split normal_total into before/after
    nb = max(min_normal_before_rows, int(round(normal_total * 0.50))) if normal_total else 0
    na = normal_total - nb

    # enforce min after if possible
    if na < min_normal_after_rows and normal_total > 0:
        shift = min(min_normal_after_rows - na, max(0, nb - min_normal_before_rows))
        nb -= shift
        na += shift

    # final safety clamp
    nb = max(0, nb)
    na = max(0, na)

    # recompute totals (must sum exactly)
    episode_total_check = nb + buildup + failure + rec + na
    if episode_total_check != episode_total:
        # adjust normal_after to make exact
        na += (episode_total - episode_total_check)
        na = max(0, na)

    return {
        "episode_total": int(nb + buildup + failure + rec + na),
        "normal_before": int(nb),
        "buildup": int(buildup),
        "failure": int(failure),
        "recovery": int(rec),
        "normal_after": int(na),
        "template_episode_id": int(template.get("meta__episode_id", -1)),
        "template_recovery_percent": float(template.get("recovery_percent", float("nan"))) if "recovery_percent" in template else None,
    }

In [14]:
# --- Load profiles (base + dropped) and build generator ---

profile_dir = Path(profile_normal_path).parent

dropped_profile_normal = profile_dir / f"{DATASET_NAME}__silver_eda__dropped_feature_profiles__normal.csv"
dropped_profile_abnormal = profile_dir / f"{DATASET_NAME}__silver_eda__dropped_feature_profiles__abnormal.csv"
dropped_profile_recovery = profile_dir / f"{DATASET_NAME}__silver_eda__dropped_feature_profiles__recovery.csv"

dropped_profile_normal_path = str(dropped_profile_normal) if dropped_profile_normal.exists() else None
dropped_profile_abnormal_path = str(dropped_profile_abnormal) if dropped_profile_abnormal.exists() else None
dropped_profile_recovery_path = str(dropped_profile_recovery) if dropped_profile_recovery.exists() else None

normal_profiles = load_and_merge_rich_profiles(
    base_profile_csv_path=str(profile_normal_path),
    state_scope="normal",
    dropped_profile_csv_path=dropped_profile_normal_path,
)

abnormal_profiles = load_and_merge_rich_profiles(
    base_profile_csv_path=str(profile_abnormal_path),
    state_scope="abnormal",
    dropped_profile_csv_path=dropped_profile_abnormal_path,
)

recovery_profiles = load_and_merge_rich_profiles(
    base_profile_csv_path=str(profile_recovery_path),
    state_scope="recovery",
    dropped_profile_csv_path=dropped_profile_recovery_path,
)

print(
    "Dropped profile files found:",
    dropped_profile_normal.exists(),
    dropped_profile_abnormal.exists(),
    dropped_profile_recovery.exists(),
)

corr_pairs_df = load_correlation_pairs_csv(corr_pairs_normal_path)
group_map_df = load_group_map_csv(group_map_normal_path)
fault_pairings_df = load_fault_pairings_csv(fault_pairings_normal_path)


def build_state_calibration_targets_from_profile_dicts(
    *,
    normal_profiles: dict,
    abnormal_profiles: dict,
    recovery_profiles: dict,
) -> dict:
    def _to_target_dict(profile_dict: dict) -> dict:
        out = {}
        for sensor, prof in profile_dict.items():
            out[str(sensor)] = {
                "mean": float(prof.mean),
                "std": float(prof.std) if pd.notna(prof.std) else 0.0,
            }
        return out

    return {
        "normal": _to_target_dict(normal_profiles),
        "abnormal": _to_target_dict(abnormal_profiles),
        "recovery": _to_target_dict(recovery_profiles),
    }


CAL_CFG = SYN_CFG.get("calibration", {}) or {}
CALIBRATION_ENABLED = bool(CAL_CFG.get("enabled", True))
CALIBRATION_MEAN_WITHIN_K_STD = float(CAL_CFG.get("mean_within_k_std", 0.75))
CALIBRATION_STD_RATIO_BOUNDS = tuple(CAL_CFG.get("std_ratio_bounds", [0.85, 1.20]))

state_calibration_targets = (
    build_state_calibration_targets_from_profile_dicts(
        normal_profiles=normal_profiles,
        abnormal_profiles=abnormal_profiles,
        recovery_profiles=recovery_profiles,
    )
    if CALIBRATION_ENABLED
    else None
)

generator = SyntheticGenerator(
    normal_profiles=normal_profiles,
    abnormal_profiles=abnormal_profiles,
    recovery_profiles=recovery_profiles,
    correlation_pairs_dataframe=corr_pairs_df,
    group_map_dataframe=group_map_df,
    fault_pairings_dataframe=fault_pairings_df,
    random_seed=int(SYN_CFG["random_seed"]),
    missingness_spec=missingness_spec if "missingness_spec" in globals() else None,
    state_calibration_targets=state_calibration_targets,
    mean_within_k_std=CALIBRATION_MEAN_WITHIN_K_STD,
    std_ratio_bounds=CALIBRATION_STD_RATIO_BOUNDS,
)

print("Sensors:", len(generator.sensors))
print("First sensors:", generator.sensors[:10])
print("Calibration enabled:", CALIBRATION_ENABLED)
print("Calibration mean_within_k_std:", CALIBRATION_MEAN_WITHIN_K_STD)
print("Calibration std_ratio_bounds:", CALIBRATION_STD_RATIO_BOUNDS)

logger.info("Generator Run")
logger.info("Generator Sensors Count: %s", len(generator.sensors))
logger.info("Generator Sensors List: %s", generator.sensors)
logger.info("Calibration Enabled: %s", CALIBRATION_ENABLED)
logger.info("Calibration mean_within_k_std: %s", CALIBRATION_MEAN_WITHIN_K_STD)
logger.info("Calibration std_ratio_bounds: %s", CALIBRATION_STD_RATIO_BOUNDS)

2026-04-05 07:04:32,950 | INFO | capstone.synthetic | Generator Run
2026-04-05 07:04:32,952 | INFO | capstone.synthetic | Generator Sensors Count: 52
2026-04-05 07:04:32,956 | INFO | capstone.synthetic | Generator Sensors List: ['sensor_00', 'sensor_01', 'sensor_02', 'sensor_03', 'sensor_04', 'sensor_05', 'sensor_06', 'sensor_07', 'sensor_08', 'sensor_09', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21', 'sensor_22', 'sensor_23', 'sensor_24', 'sensor_25', 'sensor_26', 'sensor_27', 'sensor_28', 'sensor_29', 'sensor_30', 'sensor_31', 'sensor_32', 'sensor_33', 'sensor_34', 'sensor_35', 'sensor_36', 'sensor_37', 'sensor_38', 'sensor_39', 'sensor_40', 'sensor_41', 'sensor_42', 'sensor_43', 'sensor_44', 'sensor_45', 'sensor_46', 'sensor_47', 'sensor_48', 'sensor_49', 'sensor_50', 'sensor_51']
2026-04-05 07:04:32,959 | INFO | capstone.synthetic | Calibration Enabled: True
2026-04-05 07:0

Dropped profile files found: True True True
Sensors: 52
First sensors: ['sensor_00', 'sensor_01', 'sensor_02', 'sensor_03', 'sensor_04', 'sensor_05', 'sensor_06', 'sensor_07', 'sensor_08', 'sensor_09']
Calibration enabled: True
Calibration mean_within_k_std: 0.75
Calibration std_ratio_bounds: (0.85, 1.2)


In [15]:
# --- Fault eligibility: never allow these sensors to be fault drivers/targets ---
FAULT_EXCLUDED_SENSORS = {"sensor_15", "sensor_50"}

FAULT_ELIGIBLE_SENSORS = [s for s in generator.sensors if s not in FAULT_EXCLUDED_SENSORS]

print("Fault-eligible sensors:", len(FAULT_ELIGIBLE_SENSORS))
print("Excluded sensors:", sorted(FAULT_EXCLUDED_SENSORS))

Fault-eligible sensors: 50
Excluded sensors: ['sensor_15', 'sensor_50']


In [16]:
# --- Load Silver EDA episode composition stats (normal/failure/recovery per episode) ---

episode_stats_path = profile_dir / f"{DATASET_NAME}__silver_eda__episode_status_counts.json"

EPISODE_STATS = None
if episode_stats_path.exists():
    EPISODE_STATS = json.loads(episode_stats_path.read_text())
    # optional: keep only episodes that actually contain a failure marker row
    EPISODE_STATS = [r for r in EPISODE_STATS if int(r.get("failure", 0) or 0) >= 1]
    print("Loaded episode stats rows:", len(EPISODE_STATS))
    print("episode_stats_path:", episode_stats_path)
else:
    print("No episode stats json found at:", episode_stats_path)

Loaded episode stats rows: 0
episode_stats_path: /workspace/artifacts/silver_eda/pump/pump__silver_eda__episode_status_counts.json


In [17]:


print(inspect.signature(SyntheticGenerator.__init__))

logger.info("Generator Signature Inspection: %s", inspect.signature(SyntheticGenerator.__init__))

2026-04-05 07:04:33,053 | INFO | capstone.synthetic | Generator Signature Inspection: (self, *, normal_profiles: 'Dict[str, SensorRichProfile]', abnormal_profiles: 'Dict[str, SensorRichProfile]', recovery_profiles: 'Dict[str, SensorRichProfile]', correlation_pairs_dataframe: 'pd.DataFrame', group_map_dataframe: 'pd.DataFrame', fault_pairings_dataframe: 'pd.DataFrame', random_seed: 'int' = 42, missingness_spec: 'Optional[MissingnessSpec]' = None, state_calibration_targets: 'Optional[StateCalibrationTargets]' = None, mean_within_k_std: 'float' = 1.0, std_ratio_bounds: 'Tuple[float, float]' = (0.5, 1.5)) -> 'None'


(self, *, normal_profiles: 'Dict[str, SensorRichProfile]', abnormal_profiles: 'Dict[str, SensorRichProfile]', recovery_profiles: 'Dict[str, SensorRichProfile]', correlation_pairs_dataframe: 'pd.DataFrame', group_map_dataframe: 'pd.DataFrame', fault_pairings_dataframe: 'pd.DataFrame', random_seed: 'int' = 42, missingness_spec: 'Optional[MissingnessSpec]' = None, state_calibration_targets: 'Optional[StateCalibrationTargets]' = None, mean_within_k_std: 'float' = 1.0, std_ratio_bounds: 'Tuple[float, float]' = (0.5, 1.5)) -> 'None'


In [18]:
'''
# ---- Mode switch ----
MODE = "batch"         # "single" | "batch"
TARGET_ROWS = 72_000
MAX_EPISODES = 1_000_000  # safety cap

# ---- policy knobs ----
EPISODE_MAX_ROWS = 3_000   # prevents monster episodes; forces multiple episodes in a 10k batch
ALLOW_OVERSHOOT = False    # if True, can overshoot when remaining can't fit minimum core

# ---- failure rarity control (match real dataset frequency) ----
# Real dataset: ~7 failures per 250,000 rows  => ~1 failure per 35,714 rows
ROWS_PER_FAILURE = 250_000 / 7  # ~35714.2857

'''

def _is_failure_episode(rng: np.random.Generator, *, episode_len_guess: int) -> bool:
    """
    Convert a per-row failure rate into a per-episode probability:
      P(failure in episode) ≈ episode_len / rows_per_failure
    Clipped to [0, 1].
    """
    p = float(episode_len_guess) / float(ROWS_PER_FAILURE)
    p = float(min(1.0, max(0.0, p)))
    return bool(rng.random() < p)


rng = np.random.default_rng(int(SYN_CFG.get("random_seed", 42)))
ep_cfg = SYN_CFG["episode"]

allowed_faults = (SYN_CFG.get("faults", {}) or {}).get("allowed", [])
if not allowed_faults:
    raise ValueError("SYN_CFG.faults.allowed is empty or missing")

def _safe_scalar_int(x, *, default: int = 0) -> int:
    if x is None:
        return int(default)
    if isinstance(x, (list, tuple)):
        return int(x[0]) if len(x) else int(default)
    return int(x)

def _safe_scalar_float(x, *, default: float = 0.0) -> float:
    if x is None:
        return float(default)
    if isinstance(x, (list, tuple)):
        return float(x[0]) if len(x) else float(default)
    return float(x)

def pick_fault(rng: np.random.Generator) -> str:
    if PRIMARY_FAULT_TYPE is None or str(PRIMARY_FAULT_TYPE).strip() == "":
        return str(rng.choice(allowed_faults))
    if isinstance(PRIMARY_FAULT_TYPE, (list, tuple)):
        return str(rng.choice(list(PRIMARY_FAULT_TYPE)))
    return str(PRIMARY_FAULT_TYPE)

def pick_sensor(rng: np.random.Generator) -> str:
    if PRIMARY_SENSOR is None or str(PRIMARY_SENSOR).strip() == "":
        return str(rng.choice(FAULT_ELIGIBLE_SENSORS))
    chosen = str(PRIMARY_SENSOR).strip()
    if chosen in FAULT_EXCLUDED_SENSORS:
        raise ValueError(f"PRIMARY_SENSOR='{chosen}' is excluded from fault generation.")
    return chosen

def sample_episode_spec(rng: np.random.Generator) -> EpisodeSpec:
    """Unconstrained episode sampling (good for MODE='single')."""
    magnitude = sample_float_range(rng, ep_cfg["magnitude_range"])

    is_fail = _is_failure_episode(rng)
    failure = 1 if is_fail else 0

    # Keep semantics clean:
    # - buildup only when we are going to fail
    # - recovery only when we failed
    buildup = (
        max(1, _safe_scalar_int(sample_int_range(rng, ep_cfg["buildup_range"]), default=1))
        if is_fail
        else 0
    )

    recovery = (
        max(1, _safe_scalar_int(sample_int_range(rng, ep_cfg["recovery_range"]), default=1))
        if is_fail
        else 0
    )

    return EpisodeSpec(
        primary_sensor=pick_sensor(rng),
        primary_fault_type=pick_fault(rng),
        magnitude=_safe_scalar_float(magnitude, default=1.0),
        normal_before=_safe_scalar_int(sample_int_range(rng, ep_cfg["normal_before_range"]), default=0),
        buildup=int(buildup),
        failure=int(failure),
        recovery=int(recovery),
        normal_after=_safe_scalar_int(sample_int_range(rng, ep_cfg["normal_after_range"]), default=0),
    )

def sample_episode_spec_fit_remaining(rng: np.random.Generator, remaining_rows: int) -> EpisodeSpec:
    """
    Batch-safe sampling:
      1) estimate whether the next episode should be a failure episode
      2) if failure episode and EPISODE_STATS exists, use the real episode composition helper
      3) clamp to EPISODE_MAX_ROWS and remaining_rows by shrinking normal buffers only
      4) if the remaining space is too small, emit a normal-only filler episode
    """
    remaining_rows = int(max(0, remaining_rows))

    episode_len_guess = int(min(EPISODE_MAX_ROWS, max(1, remaining_rows))) if remaining_rows > 0 else int(EPISODE_MAX_ROWS)
    is_fail = _is_failure_episode(rng, episode_len_guess=episode_len_guess)

    # -------------------------
    # Failure episode path: use real episode composition when available
    # -------------------------
    if is_fail and EPISODE_STATS:
        lengths = choose_episode_phase_lengths(
            rng,
            episode_status_rows=EPISODE_STATS,
            failure_rows_default=1,
            buildup_fraction_of_normal=0.10,
            min_buildup_rows=10,
            min_normal_before_rows=50,
            min_normal_after_rows=50,
            use_real_episode_totals=True,
        )

        nb = int(lengths["normal_before"])
        bu = int(lengths["buildup"])
        fa = int(lengths["failure"])
        rc = int(lengths["recovery"])
        na = int(lengths["normal_after"])

        fixed_core = bu + fa + rc

        # If remaining space cannot fit the failure core, emit a normal-only filler
        if remaining_rows > 0 and remaining_rows < fixed_core:
            magnitude = sample_float_range(rng, ep_cfg["magnitude_range"])
            return EpisodeSpec(
                primary_sensor=pick_sensor(rng),
                primary_fault_type=pick_fault(rng),
                magnitude=_safe_scalar_float(magnitude, default=1.0),
                normal_before=int(remaining_rows),
                buildup=0,
                failure=0,
                recovery=0,
                normal_after=0,
            )

        # Cap by EPISODE_MAX_ROWS first, shrinking normals only
        total = nb + bu + fa + rc + na
        if total > EPISODE_MAX_ROWS:
            extra = total - EPISODE_MAX_ROWS
            cut_after = min(extra, na)
            na -= cut_after
            extra -= cut_after

            if extra > 0:
                cut_before = min(extra, nb)
                nb -= cut_before
                extra -= cut_before

        # Then cap by remaining rows, shrinking normals only
        total2 = nb + bu + fa + rc + na
        if remaining_rows > 0 and total2 > remaining_rows:
            extra = total2 - remaining_rows
            cut_after = min(extra, na)
            na -= cut_after
            extra -= cut_after

            if extra > 0:
                cut_before = min(extra, nb)
                nb -= cut_before
                extra -= cut_before

        magnitude = sample_float_range(rng, ep_cfg["magnitude_range"])
        return EpisodeSpec(
            primary_sensor=pick_sensor(rng),
            primary_fault_type=pick_fault(rng),
            magnitude=_safe_scalar_float(magnitude, default=1.0),
            normal_before=int(max(0, nb)),
            buildup=int(max(0, bu)),
            failure=int(max(0, fa)),
            recovery=int(max(0, rc)),
            normal_after=int(max(0, na)),
        )

    # -------------------------
    # Non-failure or no EPISODE_STATS fallback
    # -------------------------
    failure = 1 if is_fail else 0

    buildup = (
        max(1, _safe_scalar_int(sample_int_range(rng, ep_cfg["buildup_range"]), default=1))
        if is_fail
        else 0
    )

    recovery = (
        max(1, _safe_scalar_int(sample_int_range(rng, ep_cfg["recovery_range"]), default=1))
        if is_fail
        else 0
    )

    fixed_core = buildup + failure + recovery

    if remaining_rows > 0 and remaining_rows < fixed_core:
        magnitude = sample_float_range(rng, ep_cfg["magnitude_range"])
        return EpisodeSpec(
            primary_sensor=pick_sensor(rng),
            primary_fault_type=pick_fault(rng),
            magnitude=_safe_scalar_float(magnitude, default=1.0),
            normal_before=int(remaining_rows),
            buildup=0,
            failure=0,
            recovery=0,
            normal_after=0,
        )

    nb = max(0, _safe_scalar_int(sample_int_range(rng, ep_cfg["normal_before_range"]), default=0))
    na = max(0, _safe_scalar_int(sample_int_range(rng, ep_cfg["normal_after_range"]), default=0))

    if is_fail:
        nb = min(nb, 300)
        na = min(na, 300)

    total = nb + fixed_core + na
    if total > EPISODE_MAX_ROWS:
        extra = total - EPISODE_MAX_ROWS
        cut_after = min(extra, na)
        na -= cut_after
        extra -= cut_after
        if extra > 0:
            cut_before = min(extra, nb)
            nb -= cut_before
            extra -= cut_before

    total2 = nb + fixed_core + na
    if remaining_rows > 0 and total2 > remaining_rows:
        extra = total2 - remaining_rows
        cut_after = min(extra, na)
        na -= cut_after
        extra -= cut_after
        if extra > 0:
            cut_before = min(extra, nb)
            nb -= cut_before
            extra -= cut_before

    magnitude = sample_float_range(rng, ep_cfg["magnitude_range"])
    return EpisodeSpec(
        primary_sensor=pick_sensor(rng),
        primary_fault_type=pick_fault(rng),
        magnitude=_safe_scalar_float(magnitude, default=1.0),
        normal_before=int(nb),
        buildup=int(buildup),
        failure=int(failure),
        recovery=int(recovery),
        normal_after=int(na),
    )

In [19]:
# This cell is the ONLY place that creates/overwrites synthetic_df.

if MODE == "single":
    episode = sample_episode_spec(rng)
    synthetic_df = generator.generate_episode(episode).copy()

    synthetic_df["meta__episode_id"] = 0
    synthetic_df["meta__primary_sensor"] = episode.primary_sensor
    synthetic_df["meta__primary_fault_type"] = episode.primary_fault_type
    synthetic_df["meta__magnitude"] = float(episode.magnitude)

    print("MODE=single")
    print("Episode:", episode)
    print("rows:", len(synthetic_df))

elif MODE == "batch":
    chunks = []
    row_count = 0
    episode_count = 0

    while row_count < TARGET_ROWS and episode_count < MAX_EPISODES:
        remaining = TARGET_ROWS - row_count
        spec = sample_episode_spec_fit_remaining(rng, remaining_rows=remaining)

        df_ep = generator.generate_episode(spec).copy()

        df_ep["meta__episode_id"] = episode_count
        df_ep["meta__primary_sensor"] = spec.primary_sensor
        df_ep["meta__primary_fault_type"] = spec.primary_fault_type
        df_ep["meta__magnitude"] = float(spec.magnitude)

        chunks.append(df_ep)
        row_count += len(df_ep)
        episode_count += 1

        if row_count >= TARGET_ROWS:
            break

    synthetic_df = pd.concat(chunks, ignore_index=True)

    # hard trim to exact size (safe because filler episodes are "normal-only" if needed)
    if len(synthetic_df) > TARGET_ROWS:
        synthetic_df = synthetic_df.iloc[:TARGET_ROWS].copy()

    print("MODE=batch")
    print("episodes_used:", int(synthetic_df["meta__episode_id"].nunique()) if "meta__episode_id" in synthetic_df.columns else None)
    print("unique primary_fault_type:", synthetic_df["meta__primary_fault_type"].nunique() if "meta__primary_fault_type" in synthetic_df.columns else None)
    print("unique primary_sensor:", synthetic_df["meta__primary_sensor"].nunique() if "meta__primary_sensor" in synthetic_df.columns else None)
    print("rows:", len(synthetic_df))
    display(synthetic_df["stream_state"].value_counts())

else:
    raise ValueError("MODE must be 'single' or 'batch'")

MODE=batch
episodes_used: 89
unique primary_fault_type: 8
unique primary_sensor: 42
rows: 72000


stream_state
normal        71385
recovering      611
broken            4
Name: count, dtype: int64

In [20]:
print("rows:", len(synthetic_df))
print(synthetic_df["stream_state"].value_counts().sort_index())
print(synthetic_df["stream_state"].value_counts(normalize=True).sort_index())

print("broken rows:", int((synthetic_df["stream_state"] == "broken").sum()))
print("recovering rows:", int((synthetic_df["stream_state"] == "recovering").sum()))
print("episodes:", synthetic_df["meta__episode_id"].nunique())

print("phase counts:")
print(synthetic_df["phase"].value_counts().sort_index())

allowed_stream_states = {"normal", "broken", "recovering"}
actual_stream_states = set(synthetic_df["stream_state"].dropna().astype(str).unique().tolist())

print("allowed_stream_states:", allowed_stream_states)
print("actual_stream_states:", actual_stream_states)

if not actual_stream_states.issubset(allowed_stream_states):
    raise ValueError(
        f"Unexpected stream_state values found: {sorted(actual_stream_states - allowed_stream_states)}"
    )

rows: 72000
stream_state
broken            4
normal        71385
recovering      611
Name: count, dtype: int64
stream_state
broken        0.000056
normal        0.991458
recovering    0.008486
Name: proportion, dtype: float64
broken rows: 4
recovering rows: 611
episodes: 89
phase counts:
phase
abnormal        4
buildup      1257
normal      70128
recovery      611
Name: count, dtype: int64
allowed_stream_states: {'broken', 'normal', 'recovering'}
actual_stream_states: {'broken', 'normal', 'recovering'}


In [21]:
print(synthetic_df["stream_state"].value_counts(normalize=True).sort_index())
print(synthetic_df["stream_state"].value_counts().sort_index())

stream_state
broken        0.000056
normal        0.991458
recovering    0.008486
Name: proportion, dtype: float64
stream_state
broken            4
normal        71385
recovering      611
Name: count, dtype: int64


In [22]:
print("Columns:", int(len(synthetic_df.columns)))
print("Rows:", int(len(synthetic_df)))

if "sensor_50" in synthetic_df.columns:
    print("Null % sensor_50:", float(synthetic_df["sensor_50"].isna().mean() * 100.0))
else:
    print("Null % sensor_50: NA (column missing)")

if "sensor_15" in synthetic_df.columns:
    print("All-null sensor_15:", bool(synthetic_df["sensor_15"].isna().all()))
else:
    print("All-null sensor_15: NA (column missing)")

# Basic check that we actually have abnormal segments in the output:
if "stream_state" in synthetic_df.columns:
    print("stream_state counts:", synthetic_df["stream_state"].value_counts().to_dict())

Columns: 58
Rows: 72000
Null % sensor_50: 0.0
All-null sensor_15: True
stream_state counts: {'normal': 71385, 'recovering': 611, 'broken': 4}


In [23]:
print("machine_status / stream_state counts")
display(synthetic_df["stream_state"].value_counts(dropna=False))

phase_counts = synthetic_df["phase"].value_counts(dropna=False).to_dict()
print("phase_counts:", phase_counts)

broken_rows = int((synthetic_df["stream_state"] == "broken").sum())
recovering_rows = int((synthetic_df["stream_state"] == "recovering").sum())

print("broken_rows:", broken_rows)
print("recovering_rows:", recovering_rows)

if broken_rows > 0:
    print("recovering_rows_per_broken_row:", recovering_rows / broken_rows)
else:
    print("recovering_rows_per_broken_row: no broken rows")

machine_status / stream_state counts


stream_state
normal        71385
recovering      611
broken            4
Name: count, dtype: int64

phase_counts: {'normal': 70128, 'buildup': 1257, 'recovery': 611, 'abnormal': 4}
broken_rows: 4
recovering_rows: 611
recovering_rows_per_broken_row: 152.75


In [24]:
def table_exists(engine, *, schema: str, table_name: str) -> bool:
    sql = """
    SELECT EXISTS(
      SELECT 1
      FROM information_schema.tables
      WHERE table_schema = :schema AND table_name = :table
    ) AS exists_flag
    """
    df = read_sql_dataframe(engine, sql, params={"schema": schema, "table": table_name})
    return bool(df.loc[0, "exists_flag"])


def drop_table(engine, *, schema: str, table_name: str) -> None:
    execute_sql(engine, f'DROP TABLE IF EXISTS "{schema}"."{table_name}" CASCADE')


def get_max_batch_id(engine, *, schema: str, table_name: str, batch_col: str = "batch_id") -> int:
    fq = f'"{schema}"."{table_name}"'
    df = read_sql_dataframe(engine, f"SELECT COALESCE(MAX({batch_col}), 0) AS max_batch_id FROM {fq}")
    return int(df.loc[0, "max_batch_id"])


def canonicalize_existing_batch_ids(engine, *, schema: str, table_name: str, batch_col: str = "batch_id") -> Dict[str, int]:
    """
    Renumber existing batch ids in-place to 1..N, preserving ascending order.
    Example: [4,5,6,7] -> [1,2,3,4]
    """
    fq = f'"{schema}"."{table_name}"'

    sql = f"""
    WITH b AS (
        SELECT DISTINCT {batch_col} AS old_batch_id
        FROM {fq}
        WHERE {batch_col} IS NOT NULL
    ),
    m AS (
        SELECT old_batch_id,
               DENSE_RANK() OVER (ORDER BY old_batch_id ASC) AS new_batch_id
        FROM b
    )
    UPDATE {fq} t
    SET {batch_col} = m.new_batch_id
    FROM m
    WHERE t.{batch_col} = m.old_batch_id
    """
    execute_sql(engine, sql)

    df = read_sql_dataframe(
        engine,
        f"""
        SELECT
          COUNT(DISTINCT {batch_col}) AS distinct_batches,
          COALESCE(MIN({batch_col}), 0) AS min_batch_id,
          COALESCE(MAX({batch_col}), 0) AS max_batch_id
        FROM {fq}
        """
    )
    return {
        "distinct_batches": int(df.loc[0, "distinct_batches"]),
        "min_batch_id": int(df.loc[0, "min_batch_id"]),
        "max_batch_id": int(df.loc[0, "max_batch_id"]),
    }


    
def choose_batch_id(
    engine,
    *,
    schema: str,
    table_name: str,
    write_mode: str,    # "reset" | "append"
    append_mode: str,   # "continue" | "renumber" (only used if write_mode="append")
    batch_col: str = "batch_id",
) -> int:
    """
    Implements your exact behavior:

    - write_mode="reset":
        drop table, recreate later via writer, next batch_id = 1

    - write_mode="append", append_mode="continue":
        do NOT renumber; next batch_id = MAX(batch_id)+1

    - write_mode="append", append_mode="renumber":
        renumber existing batches to 1..N (in-place), then next batch_id = N+1
        Example existing [4,5,6,7], new insert becomes 5 while table becomes [1,2,3,4,5]
    """
    write_mode = str(write_mode).strip().lower()
    append_mode = str(append_mode).strip().lower()

    if write_mode == "reset":
        # hard reset: drop table; next batch is 1
        drop_table(engine, schema=schema, table_name=table_name)
        return 1

    if write_mode != "append":
        raise ValueError("write_mode must be 'reset' or 'append'")

    exists = table_exists(engine, schema=schema, table_name=table_name)
    if not exists:
        return 1

    if append_mode == "continue":
        return get_max_batch_id(engine, schema=schema, table_name=table_name, batch_col=batch_col) + 1

    if append_mode == "renumber":
        summary = canonicalize_existing_batch_ids(engine, schema=schema, table_name=table_name, batch_col=batch_col)
        return int(summary["max_batch_id"]) + 1

    raise ValueError("append_mode must be 'continue' or 'renumber'")

In [ ]:
engine = get_engine_from_env()

'''
PG_SCHEMA = "capstone"
ARTIFACT_NAME = "stream"
TABLE_NAME = f"synthetic_{DATASET_NAME.lower()}_{ARTIFACT_NAME}"

# ---- write mode flags
WRITE_MODE = "reset"       # "reset" | "append"
APPEND_MODE = "renumber"    # "continue" | "renumber"   (only matters if WRITE_MODE="append")

'''
batch_id = choose_batch_id(
    engine,
    schema=PG_SCHEMA,
    table_name=TABLE_NAME,
    write_mode=WRITE_MODE,
    append_mode=APPEND_MODE,
    batch_col="batch_id",
)

print("Chosen batch_id:", batch_id)


if str(WRITE_MODE).strip().lower() == "reset":
    ensure_sequence(engine, schema=PG_SCHEMA, sequence_name=f"seq_synthetic_{DATASET_NAME.lower()}_batch_id")
    ensure_sequence(engine, schema=PG_SCHEMA, sequence_name=f"seq_synthetic_{DATASET_NAME.lower()}_cycle_id")
    reset_synthetic_sequences(engine, schema=PG_SCHEMA, dataset_name=DATASET_NAME)


# cycles can stay sequence-based for now
ensure_sequence(engine, schema=PG_SCHEMA, sequence_name=f"seq_synthetic_{DATASET_NAME.lower()}_cycle_id")


cycle_start = reserve_cycle_range(
    engine,
    schema=PG_SCHEMA,
    sequence_name=f"seq_synthetic_{DATASET_NAME.lower()}_cycle_id",
    n_rows=len(synthetic_df),
)

table_written = write_stream_batch(
    engine,
    synthetic_df,
    dataset_name=DATASET_NAME,
    schema=PG_SCHEMA,
    artifact_name=ARTIFACT_NAME,
    batch_id=batch_id,
    cycle_start=cycle_start,
)

print("Wrote:", table_written, "batch_id:", batch_id, "cycle_start:", cycle_start)

Chosen batch_id: 1
[synthetic] Added 56 new columns to capstone.synthetic_pump_stream


In [ ]:
import os
{k: os.environ.get(k) for k in ["DB_HOST","DB_PORT","DB_NAME","DB_USER","DB_PASSWORD","POSTGRES_DB","POSTGRES_USER","POSTGRES_PASSWORD"]}

{'DB_HOST': 'dcook_capstone_postgres',
 'DB_PORT': '5432',
 'DB_NAME': 'dcook_capstone_postgres_db',
 'DB_USER': 'dcook_admin',
 'DB_PASSWORD': 'V9m!pQ4z@H2eS7wK',
 'POSTGRES_DB': None,
 'POSTGRES_USER': None,
 'POSTGRES_PASSWORD': None}

In [ ]:
import importlib.util
print("psycopg:", importlib.util.find_spec("psycopg"))
print("psycopg2:", importlib.util.find_spec("psycopg2"))

psycopg: None
psycopg2: ModuleSpec(name='psycopg2', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7600ac12d4b0>, origin='/opt/miniconda/envs/capstone/lib/python3.10/site-packages/psycopg2/__init__.py', submodule_search_locations=['/opt/miniconda/envs/capstone/lib/python3.10/site-packages/psycopg2'])


In [ ]:
def _as_int(x):
    """
    Coerce x into an int if possible.
    - If x is list/tuple: take the first element (or None if empty)
    - If x is None: return None
    - If x is float/string/int: convert safely
    """
    if x is None:
        return None
    if isinstance(x, (list, tuple)):
        if len(x) == 0:
            return None
        x = x[0]
    try:
        return int(x)
    except Exception:
        return None


def _as_float(x):
    """
    Coerce x into a float if possible.
    - If x is list/tuple: take the first element (or None if empty)
    - If x is None: return None
    """
    if x is None:
        return None
    if isinstance(x, (list, tuple)):
        if len(x) == 0:
            return None
        x = x[0]
    try:
        return float(x)
    except Exception:
        return None

In [ ]:
# episodes actually created
print("episodes:", synthetic_df["meta__episode_id"].nunique() if "meta__episode_id" in synthetic_df.columns else "NO meta__episode_id")

# how many different primary faults / sensors happened across episodes
if "meta__primary_fault_type" in synthetic_df.columns:
    print("unique primary_fault_type:", synthetic_df["meta__primary_fault_type"].nunique())
    print(synthetic_df[["meta__episode_id","meta__primary_fault_type"]].drop_duplicates().head(10))

if "meta__primary_sensor" in synthetic_df.columns:
    print("unique primary_sensor:", synthetic_df["meta__primary_sensor"].nunique())
    print(synthetic_df[["meta__episode_id","meta__primary_sensor"]].drop_duplicates().head(10))

episodes: 85
unique primary_fault_type: 8
      meta__episode_id meta__primary_fault_type
0                    0               step_shift
837                  1     intermittent_dropout
1839                 2                 sawtooth
2755                 3               drift_down
3645                 4               step_shift
4426                 5                 sawtooth
5248                 6                 drift_up
6030                 7                 sawtooth
7001                 8               drift_down
7912                 9               step_shift
unique primary_sensor: 42
      meta__episode_id meta__primary_sensor
0                    0            sensor_04
837                  1            sensor_36
1839                 2            sensor_09
2755                 3            sensor_23
3645                 4            sensor_13
4426                 5            sensor_23
5248                 6            sensor_25
6030                 7            sensor_19
7001    

In [ ]:
process_run_id = make_process_run_id(str(SYN_CFG.get("process_run_id_prefix", "synthetic")))

synthetic_truth = initialize_layer_truth(
    truth_version=str(TRUTH_VERSION),
    dataset_name=DATASET_NAME,
    layer_name="synthetic",
    process_run_id=process_run_id,
    pipeline_mode=PIPELINE_MODE,
    parent_truth_hash=PARENT_TRUTH_HASH,
)

LAYER_NAME = "synthetic"

resolved_config_dir = ARTIFACTS_ROOT / "synthetic" / DATASET_NAME
resolved_config_dir.mkdir(parents=True, exist_ok=True)

resolved_config_path = resolved_config_dir / f"{DATASET_NAME}__{LAYER_NAME}__resolved_config.yaml"

# export_config_snapshot requires a destination path
export_config_snapshot(CONFIG, destination=resolved_config_path)

print("Resolved config written to:", resolved_config_path)


synthetic_truth = update_truth_section(
    synthetic_truth,
    "config_snapshot",
    {
        # store the path you just wrote
        "resolved_config_path": str(resolved_config_path),

        # optional: small inline config view (JSON-safe)
        "truth_config_block": build_truth_config_block(CONFIG),

        # keep your synthetic config subset if you want
        "synthetic_cfg": SYN_CFG,
    },
)

synthetic_truth = update_truth_section(
    synthetic_truth,
    "runtime_facts",
    {
        "primary_sensor": (
            getattr(episode, "primary_sensor", None)
            if "episode" in globals()
            else (str(synthetic_df["meta__primary_sensor"].mode(dropna=True).iloc[0]) if "meta__primary_sensor" in synthetic_df.columns else None)
        ),
        "primary_fault_type": (
            getattr(episode, "primary_fault_type", None)
            if "episode" in globals()
            else (str(synthetic_df["meta__primary_fault_type"].mode(dropna=True).iloc[0]) if "meta__primary_fault_type" in synthetic_df.columns else None)
        ),
        "episode": (
            dict(episode.__dict__)
            if ("episode" in globals() and episode is not None and hasattr(episode, "__dict__"))
            else {
                "mode": str(MODE) if "MODE" in globals() else None,
                "episodes_used": int(episode_count) if "episode_count" in globals() else None,
                "target_rows": int(TARGET_ROWS) if "TARGET_ROWS" in globals() else None,
                "meta_episode_id_min": int(synthetic_df["meta__episode_id"].min()) if "meta__episode_id" in synthetic_df.columns else None,
                "meta_episode_id_max": int(synthetic_df["meta__episode_id"].max()) if "meta__episode_id" in synthetic_df.columns else None,
                "meta_primary_sensor_mode": (
                    str(synthetic_df["meta__primary_sensor"].mode(dropna=True).iloc[0])
                    if "meta__primary_sensor" in synthetic_df.columns and not synthetic_df["meta__primary_sensor"].dropna().empty
                    else None
                ),
                "meta_primary_fault_type_mode": (
                    str(synthetic_df["meta__primary_fault_type"].mode(dropna=True).iloc[0])
                    if "meta__primary_fault_type" in synthetic_df.columns and not synthetic_df["meta__primary_fault_type"].dropna().empty
                    else None
                ),
            }
        ),
        "normal_before_range": ep_cfg["normal_before_range"],
        "normal_before_selection": _as_int(NORMAL_BEFORE) if "NORMAL_BEFORE" in globals() else None,
        "buildup_range": ep_cfg["buildup_range"],
        "buildup_selection": _as_int(BUILDUP) if "BUILDUP" in globals() else None,
        "failure_range": ep_cfg["failure_range"],
        "failure_selection": _as_int(FAILURE) if "FAILURE" in globals() else None,
        "recovery_range": ep_cfg["recovery_range"],
        "recovery_selection": _as_int(RECOVERY) if "RECOVERY" in globals() else None,
        "normal_after_range": ep_cfg["normal_after_range"],
        "normal_after_selection": _as_int(NORMAL_AFTER) if "NORMAL_AFTER" in globals() else None,
        "magnitude_range": ep_cfg["magnitude_range"],
        "magnitude_selection": _as_float(MAGNITUDE) if "MAGNITUDE" in globals() else None,
        "row_count": int(len(synthetic_df)),
        "parent_truth_hash": PARENT_TRUTH_HASH,
        "silver_eda_truth_hash": PARENT_TRUTH_HASH,
    },
)

# Optionally save local parquet artifact too (useful for debugging)
synth_dir = ARTIFACTS_ROOT / "synthetic" / DATASET_NAME
synth_dir.mkdir(parents=True, exist_ok=True)
suffix = "episode" if ("MODE" in globals() and str(MODE) == "single") else "batch"
out_path = synth_dir / f"{DATASET_NAME}__synthetic__{suffix}.parquet"
save_data(synthetic_df, synth_dir, out_path.name)

artifact_paths_payload = {
    "silver_eda_truth_hash": PARENT_TRUTH_HASH,
    "profile_normal_path": str(profile_normal_path),
    "profile_abnormal_path": str(profile_abnormal_path),
    "profile_recovery_path": str(profile_recovery_path),
    "corr_pairs_normal_path": corr_pairs_normal_path,
    "group_map_normal_path": group_map_normal_path,
    "fault_pairings_normal_path": fault_pairings_normal_path,
    "synthetic_episode_path": str(out_path),
    "postgres_schema": PG_SCHEMA,
    "postgres_table": (
        table_written if "table_written" in globals() else None
    ),  
    "postgres_batch_id": int(batch_id),
    "postgres_cycle_start": int(cycle_start),
}

#export_path = Path(PATHS["data_raw_dir"] / "synthetic") 

if EXPORT_ENABLED:
    artifact_paths_payload["export_batch_parquet_path"] = str(out_path)

synthetic_truth = update_truth_section(synthetic_truth, "artifact_paths", artifact_paths_payload)

meta_columns = sorted(["meta__truth_hash", "meta__parent_truth_hash", "meta__pipeline_mode"])
feature_columns = sorted([column for column in synthetic_df.columns if not str(column).startswith("meta__")])

truth_record = build_truth_record(
    truth_base=synthetic_truth,
    row_count=int(len(synthetic_df)),
    column_count=int(synthetic_df.shape[1] + 3),
    meta_columns=meta_columns,
    feature_columns=feature_columns,
)

synth_truth_hash = truth_record["truth_hash"]

# stamp lineage columns into dataframe (optional)
synthetic_df = stamp_truth_columns(
    synthetic_df,
    truth_hash=synth_truth_hash,
    parent_truth_hash=PARENT_TRUTH_HASH,
    pipeline_mode=PIPELINE_MODE,
)

truth_path = save_truth_record(
    truth_record,
    truth_dir=TRUTHS_PATH,
    dataset_name=DATASET_NAME,
    layer_name="synthetic",
)

append_truth_index(truth_record, truth_index_path=TRUTH_INDEX_PATH)

print("Synthetic truth hash:", synth_truth_hash)
print("Synthetic truth path:", truth_path)
print("Local episode parquet:", out_path)

Resolved config written to: /workspace/artifacts/synthetic/pump/pump__synthetic__resolved_config.yaml


2026-04-04 03:56:52,739 | INFO | capstone.file_io | Saving DataFrame to Parquet: /workspace/artifacts/synthetic/pump/pump__synthetic__batch.parquet


2026-04-04 03:56:53,616 | INFO | capstone.file_io | Saved: pump__synthetic__batch.parquet | shape=(72000, 58) | columns=['sensor_00', 'sensor_01', 'sensor_02', 'sensor_03', 'sensor_04', 'sensor_05', 'sensor_06', 'sensor_07', 'sensor_08', 'sensor_09', 'sensor_10', 'sensor_11', 'sensor_12', 'sensor_13', 'sensor_14', 'sensor_15', 'sensor_16', 'sensor_17', 'sensor_18', 'sensor_19', 'sensor_20', 'sensor_21', 'sensor_22', 'sensor_23', 'sensor_24', 'sensor_25', 'sensor_26', 'sensor_27', 'sensor_28', 'sensor_29', 'sensor_30', 'sensor_31', 'sensor_32', 'sensor_33', 'sensor_34', 'sensor_35', 'sensor_36', 'sensor_37', 'sensor_38', 'sensor_39', 'sensor_40', 'sensor_41', 'sensor_42', 'sensor_43', 'sensor_44', 'sensor_45', 'sensor_46', 'sensor_47', 'sensor_48', 'sensor_49', 'sensor_50', 'sensor_51', 'stream_state', 'phase', 'meta__episode_id', 'meta__primary_sensor', 'meta__primary_fault_type', 'meta__magnitude']


Synthetic truth hash: 8d399c2c93b05d7e8c2c8a4bf494f553d520f0ba6ef8ef9e99b9fa6e33289a27
Synthetic truth path: /workspace/artifacts/truths/synthetic/pump__synthetic__truth__8d399c2c93b05d7e8c2c8a4bf494f553d520f0ba6ef8ef9e99b9fa6e33289a27.json
Local episode parquet: /workspace/artifacts/synthetic/pump/pump__synthetic__batch.parquet


In [ ]:
from utils.synthetic.generator.synthetic_profiles import load_and_merge_rich_profiles
from utils.synthetic.generator.synthetic_generator import SyntheticGenerator, EpisodeSpec
from utils.synthetic.generator.synthetic_missingness import build_missingness_spec_from_truth_payload
from utils.synthetic.generator.synthetic_export import export_synthetic_batch_to_parquet

print("Imports OK")

Imports OK


In [ ]:
print("Episodes:", synthetic_df["meta__episode_id"].nunique() if "meta__episode_id" in synthetic_df else "NA")
print("Primary fault types:", synthetic_df["meta__primary_fault_type"].value_counts().head(10) if "meta__primary_fault_type" in synthetic_df else "NA")
print("Primary sensors:", synthetic_df["meta__primary_sensor"].value_counts().head(10) if "meta__primary_sensor" in synthetic_df else "NA")

Episodes: 85
Primary fault types: meta__primary_fault_type
sawtooth                14909
drift_up                 9471
intermittent_dropout     9436
variance_burst           9423
spike                    8260
step_shift               7925
drift_down               7902
stuck_constant           4674
Name: count, dtype: int64
Primary sensors: meta__primary_sensor
sensor_13    4039
sensor_51    3975
sensor_37    3933
sensor_36    3398
sensor_40    3190
sensor_17    2992
sensor_07    2855
sensor_03    2622
sensor_04    2615
sensor_45    2592
Name: count, dtype: int64


In [ ]:
def verify_schema(
    df,
    *,
    expected_sensors: list[str],
    state_col: str = "stream_state",
    phase_col: str = "phase",
) -> dict:
    cols = set(df.columns)
    exp = set(expected_sensors)

    missing_cols = sorted(exp - cols)
    extra_sensor_cols = sorted([c for c in cols if c.startswith("sensor_") and c not in exp])

    out = {
        "row_count": int(len(df)),
        "missing_sensor_columns": missing_cols,
        "extra_sensor_columns": extra_sensor_cols,
        "state_values": sorted(df[state_col].dropna().astype(str).unique()) if state_col in df.columns else None,
        "phase_values": sorted(df[phase_col].dropna().astype(str).unique()) if phase_col in df.columns else None,
    }
    return out

In [ ]:
def verify_missingness_exact(
    df: pd.DataFrame,
    *,
    target_missing_pct: dict[str, float],   # sensor -> percent (0..100)
    sensors: list[str],
    tol_missing_rows: int = 0,              # 0 = strict integer exactness
) -> pd.DataFrame:
    n = int(len(df))
    rows = []

    for s in sensors:
        if s not in df.columns:
            continue

        target = float(target_missing_pct.get(s, 0.0))

        # integer expectation consistent with generator logic
        expected_present = int(round(n * (1.0 - target / 100.0)))
        expected_present = max(0, min(n, expected_present))
        expected_missing = n - expected_present

        actual_missing = int(df[s].isna().sum())
        diff_rows = actual_missing - expected_missing

        ok = abs(diff_rows) <= int(tol_missing_rows)

        rows.append({
            "sensor": s,
            "target_pct": target,
            "n_rows": n,
            "expected_missing_rows": expected_missing,
            "actual_missing_rows": actual_missing,
            "diff_missing_rows": diff_rows,
            "ok": ok,
            # optional diagnostic, but not used for pass/fail:
            "actual_pct": float(actual_missing / n * 100.0) if n > 0 else np.nan,
        })

    out = pd.DataFrame(rows)
    if out.empty:
        return out
    return out.sort_values("diff_missing_rows", key=lambda c: c.abs(), ascending=False)

In [ ]:
def verify_profile_bounds(
    df: pd.DataFrame,
    *,
    profile_df: pd.DataFrame,     # rows: sensor,state_scope,p01,p99,mean,std,...
    sensors: list[str],
    state_col: str = "stream_state",
    state_values: list[str] = ["normal","abnormal","recovery"],
    bound_cols: tuple[str, str] = ("p01","p99"),
) -> pd.DataFrame:
    p01_col, p99_col = bound_cols
    prof = profile_df.copy()
    prof["sensor"] = prof["sensor"].astype(str)
    prof["state_scope"] = prof["state_scope"].astype(str)

    rows = []
    for st in state_values:
        df_st = df.loc[df[state_col].astype(str) == st]
        if df_st.empty:
            continue

        for s in sensors:
            if s not in df_st.columns:
                continue

            # profile row
            p = prof.loc[(prof["sensor"] == s) & (prof["state_scope"] == st)]
            if p.empty:
                continue
            p = p.iloc[0]

            series = pd.to_numeric(df_st[s], errors="coerce")
            nonnull = series.dropna()
            if nonnull.empty:
                rows.append({"state": st, "sensor": s, "nonnull_n": 0, "pct_outside_bounds": np.nan})
                continue

            lo = float(p[p01_col])
            hi = float(p[p99_col])

            outside = ((nonnull < lo) | (nonnull > hi)).mean() * 100.0
            rows.append({
                "state": st,
                "sensor": s,
                "nonnull_n": int(nonnull.shape[0]),
                "p01": lo,
                "p99": hi,
                "pct_outside_bounds": float(outside),
                "mean_actual": float(nonnull.mean()),
                "std_actual": float(nonnull.std(ddof=1)) if nonnull.shape[0] > 1 else 0.0,
                "mean_profile": float(p["mean"]) if "mean" in p else np.nan,
                "std_profile": float(p["std"]) if "std" in p else np.nan,
            })

    out = pd.DataFrame(rows)
    if out.empty:
        return out
    return out.sort_values(["pct_outside_bounds","nonnull_n"], ascending=[False, True])

In [ ]:
def verify_top_correlations(
    df: pd.DataFrame,
    *,
    corr_pairs_df: pd.DataFrame,      # sensor_a,sensor_b,abs_pearson_corr,...
    sensors: list[str],
    state_col: str = "stream_state",
    state_value: str = "normal",
    top_k: int = 25,
) -> pd.DataFrame:
    df_n = df.loc[df[state_col].astype(str) == state_value, sensors].copy()
    df_n = df_n.apply(pd.to_numeric, errors="coerce").dropna(axis=1, how="all")
    pear = df_n.corr(method="pearson")

    pairs = corr_pairs_df.head(top_k).copy()
    rows = []
    for _, r in pairs.iterrows():
        a, b = str(r["sensor_a"]), str(r["sensor_b"])
        if a in pear.columns and b in pear.columns:
            actual = float(pear.loc[a, b])
            rows.append({
                "sensor_a": a,
                "sensor_b": b,
                "expected_abs": float(r.get("abs_pearson_corr", np.nan)),
                "actual_abs": float(abs(actual)),
                "actual_signed": actual,
            })
    return pd.DataFrame(rows).sort_values("expected_abs", ascending=False)

In [ ]:
# ---------------------------
# 0) Choose your sensor list
# ---------------------------
SENSORS = list(generator.sensors)  # best source of truth
# If you prefer the dataframe view:
# SENSORS = sorted([c for c in synthetic_df.columns if c.startswith("sensor_")])

# ---------------------------
# 1) Schema / state sanity
# ---------------------------
schema_report = verify_schema(
    synthetic_df,
    expected_sensors=SENSORS,
    state_col="stream_state",
    phase_col="phase",
)
print("Schema report:", schema_report)

# ---------------------------
# 2) Missingness sanity
#    (GLOBAL target; easiest / most stable)
# ---------------------------
# Option A: from your MissingnessSpec (preferred if you use it)
target_missing_pct = None
if getattr(generator, "missingness_spec", None) is not None:
    target_missing_pct = dict(generator.missingness_spec.missingness_pct_all)

print("Has missingness_spec?:", getattr(generator, "missingness_spec", None) is not None)
print("target_missing_pct size:", 0 if target_missing_pct is None else len(target_missing_pct))

# Option B: from your Silver truth
'''
target_missing_pct = (
    (silver_truth.get("runtime_facts", {}) or {})
    .get("missingness_quarantine", {})
    .get("missingness_pct_all")
)

print("target_missing_pct size:", 0 if not target_missing_pct else len(target_missing_pct))
'''

if target_missing_pct is None:
    print("Skipping missingness check: no target_missing_pct available")
else:
    # Force known all-null sensors to 100% missing (matches Silver PreEDA quarantine reality)
    if isinstance(target_missing_pct, dict):
        target_missing_pct = dict(target_missing_pct)  # defensive copy
        target_missing_pct["sensor_15"] = 100.0
        
    missing_report = verify_missingness_exact(
        synthetic_df,
        target_missing_pct=target_missing_pct,
        sensors=generator.sensors,
        tol_missing_rows=0,   # strict
    )

    display(missing_report.head(30))
    print("Missingness OK?:", bool(missing_report["ok"].all()))

# ---------------------------
# 3) Profile bounds sanity
#    Build one combined profile_df (sensor,state_scope,p01,p99,mean,std,...)
# ---------------------------
def _profiles_dict_to_df(d: dict, state: str) -> pd.DataFrame:
    rows = []
    for sensor, prof in d.items():
        # SensorRichProfile is dataclass-like; __dict__ works
        r = dict(prof.__dict__)
        r["sensor"] = str(sensor)
        r["state_scope"] = str(state)
        rows.append(r)
    return pd.DataFrame(rows)

profile_df = pd.concat(
    [
        _profiles_dict_to_df(normal_profiles, "normal"),
        _profiles_dict_to_df(abnormal_profiles, "abnormal"),
        _profiles_dict_to_df(recovery_profiles, "recovery"),
    ],
    ignore_index=True,
)

print("profile_df columns:", sorted(profile_df.columns.tolist()))
display(profile_df.head(5))



# --- For verification only: map buildup -> abnormal ---
df_check = synthetic_df.copy()
df_check["bounds_state"] = df_check["stream_state"].astype(str)
df_check.loc[df_check["bounds_state"].eq("buildup"), "bounds_state"] = "abnormal"

bounds_check = verify_profile_bounds(
    df_check,
    profile_df=profile_df,
    sensors=SENSORS,
    state_col="bounds_state",
    state_values=["normal", "abnormal", "recovery"],
    bound_cols=("p01", "p99"),
)

display(bounds_check.head(50))


# rule of thumb: if pct_outside_bounds is consistently > ~1-3%,
# your generator clipping or distributions are drifting from profile expectations.

# ---------------------------
# 4) Correlation sanity (normal only)
# ---------------------------
corr_check = verify_top_correlations(
    synthetic_df,
    corr_pairs_df=corr_pairs_df,
    sensors=SENSORS,
    state_col="stream_state",
    state_value="normal",
    top_k=25,
)
display(corr_check)

print("Done.")

Schema report: {'row_count': 72000, 'missing_sensor_columns': [], 'extra_sensor_columns': [], 'state_values': ['broken', 'normal', 'recovering'], 'phase_values': ['abnormal', 'buildup', 'normal', 'recovery']}
Has missingness_spec?: True
target_missing_pct size: 50


,sensor,target_pct,n_rows,expected_missing_rows,actual_missing_rows,diff_missing_rows,ok,actual_pct
29,sensor_29,0.032680,72000,24,0,-24,False,0.0
32,sensor_32,0.030864,72000,22,0,-22,False,0.0
18,sensor_18,0.020879,72000,15,0,-15,False,0.0
17,sensor_17,0.020879,72000,15,0,-15,False,0.0
22,sensor_22,0.018609,72000,13,0,-13,False,0.0
25,sensor_25,0.016340,72000,12,0,-12,False,0.0
16,sensor_16,0.014070,72000,10,0,-10,False,0.0
40,sensor_40,0.012255,72000,9,0,-9,False,0.0
39,sensor_39,0.012255,72000,9,0,-9,False,0.0
38,sensor_38,0.012255,72000,9,0,-9,False,0.0


Missingness OK?: False
profile_df columns: ['distribution_family', 'iqr', 'kurtosis', 'lower_bound', 'max_value', 'mean', 'median', 'min_value', 'p01', 'p05', 'p25', 'p50', 'p75', 'p95', 'p99', 'robust_std', 'sensor', 'skewness', 'state_scope', 'std', 'upper_bound']


,sensor,state_scope,mean,std,min_value,max_value,median,iqr,p01,p05,...,p50,p75,p95,p99,skewness,kurtosis,robust_std,distribution_family,lower_bound,upper_bound
0,sensor_28,normal,834.805447,315.731104,4.319347,1841.1460,940.18630,246.180600,51.936918,78.948944,...,940.18630,1019.285250,1153.126150,1260.008310,-1.281339,1.054851,182.491179,left_skewed,51.936918,1260.008310
1,sensor_23,normal,915.207197,299.415820,0.000000,1227.5640,981.54525,139.781875,64.946174,95.083560,...,981.54525,1090.722000,1098.354050,1110.298000,-2.149939,3.105727,103.618884,left_skewed,64.946174,1110.298000
2,sensor_36,normal,594.512709,293.535541,2.260970,983.9208,771.91810,574.141050,56.462810,95.837888,...,771.91810,838.312425,877.123445,914.674382,-0.626473,-1.289883,425.604930,bounded_normal,56.462810,914.674382
3,sensor_31,normal,851.400569,285.366194,23.958330,1800.0000,911.45830,144.270800,76.041660,138.541700,...,911.45830,974.999900,1085.417000,1800.000000,-0.950618,2.880065,106.946479,bounded_normal,76.041660,1800.000000
4,sensor_32,normal,794.855086,261.595990,0.240716,1839.2110,876.64740,198.173450,91.755291,131.014785,...,876.64740,944.530175,1013.162650,1208.683740,-1.276655,2.335709,146.903966,left_skewed,91.755291,1208.683740


,state,sensor,nonnull_n,p01,p99,pct_outside_bounds,mean_actual,std_actual,mean_profile,std_profile
51,normal,sensor_51,63992,46.585650,294.849500,0.0,216.912553,50.818080,200.720874,62.304341
0,normal,sensor_00,65611,0.683738,2.519502,0.0,2.320537,0.199018,2.420476,0.242472
7,normal,sensor_07,67092,14.684610,17.621530,0.0,15.978978,0.506874,16.165959,0.600457
8,normal,sensor_08,67198,13.990160,17.158560,0.0,15.297514,0.496396,15.478129,0.588606
6,normal,sensor_06,67297,11.224177,15.198210,0.0,13.586729,0.731253,13.865073,0.864943
9,normal,sensor_09,67363,13.809320,16.579860,0.0,14.910224,0.481747,15.086662,0.572969
1,normal,sensor_01,68684,43.229160,54.210070,0.0,48.178630,1.893690,48.180116,2.240173
30,normal,sensor_30,68711,72.222220,931.138854,0.0,545.226572,169.058965,608.653372,200.362019
2,normal,sensor_02,68796,45.538624,54.904510,0.0,51.024242,1.590686,51.636949,1.878299
3,normal,sensor_03,68796,39.496520,47.265625,0.0,44.153812,1.399156,44.167060,1.668260


,sensor_a,sensor_b,expected_abs,actual_abs,actual_signed
0,sensor_19,sensor_20,NaN,0.021892,0.021892
1,sensor_20,sensor_21,NaN,0.016348,0.016348
2,sensor_19,sensor_21,NaN,0.018443,0.018443
3,sensor_14,sensor_16,NaN,0.011221,0.011221
4,sensor_19,sensor_22,NaN,0.019756,0.019756
5,sensor_19,sensor_24,NaN,0.027351,0.027351
6,sensor_20,sensor_22,NaN,0.017755,0.017755
7,sensor_21,sensor_24,NaN,0.024802,0.024802
8,sensor_22,sensor_23,NaN,0.022048,0.022048
9,sensor_20,sensor_24,NaN,0.022499,0.022499


Done.


In [ ]:
ep0 = synthetic_df.loc[synthetic_df["meta__episode_id"] == 0].copy()
print(ep0["stream_state"].value_counts())

primary = str(ep0["meta__primary_sensor"].dropna().iloc[0])
print("Primary sensor:", primary)

# show some candidate secondaries from your fault_pairings_df
links = fault_pairings_df.loc[fault_pairings_df["sensor_primary"].astype(str) == primary].copy()
display(links.sort_values("fault_coupling_strength", ascending=False).head(10))

stream_state
normal    837
Name: count, dtype: int64
Primary sensor: sensor_04


,sensor_primary,sensor_secondary,relationship_type,correlation_strength,fault_coupling_strength,recommended_secondary_fault,lag_cycles,same_group_flag,evidence_score,group_name_primary,group_name_secondary
111,sensor_04,sensor_00,same_subsystem_coupled,0.91524,1.0,variance_burst,0,True,1.0,group_002,group_002


In [ ]:
ep0 = synthetic_df.loc[synthetic_df["meta__episode_id"] == 0].copy()
print(ep0["stream_state"].value_counts())

primary = str(ep0["meta__primary_sensor"].dropna().iloc[0])
print("Primary sensor:", primary)

# show some candidate secondaries from your fault_pairings_df
links = fault_pairings_df.loc[fault_pairings_df["sensor_primary"].astype(str) == primary].copy()
display(links.sort_values("fault_coupling_strength", ascending=False).head(10))

stream_state
normal    837
Name: count, dtype: int64
Primary sensor: sensor_04


,sensor_primary,sensor_secondary,relationship_type,correlation_strength,fault_coupling_strength,recommended_secondary_fault,lag_cycles,same_group_flag,evidence_score,group_name_primary,group_name_secondary
111,sensor_04,sensor_00,same_subsystem_coupled,0.91524,1.0,variance_burst,0,True,1.0,group_002,group_002


In [ ]:
# These should never be selected as primary sensors
if "meta__primary_sensor" in synthetic_df.columns:
    bad = synthetic_df.loc[synthetic_df["meta__primary_sensor"].isin(["sensor_15","sensor_50"])]
    print("Bad primary sensor rows:", len(bad))

# These should be all-null / high-missing as expected
print("sensor_15 missing%:", float(synthetic_df["sensor_15"].isna().mean() * 100.0))
print("sensor_50 missing%:", float(synthetic_df["sensor_50"].isna().mean() * 100.0))

Bad primary sensor rows: 0
sensor_15 missing%: 100.0
sensor_50 missing%: 0.0


----

In [ ]:
# Dispose Engine
engine.dispose()